# 📚 Notebook 02 — 从零构建技术指标

**第一阶段：基础知识** · 前置要求：Notebook 01（OHLCV 数据）

---

## 🎯 学习目标

完成本 Notebook 后，你将能够：

1. 计算**指数移动平均线 (EMA)** 并理解平滑参数 $\alpha$
2. 使用 EWM 方法构建 **RSI**（相对强弱指数）— 与生产代码完全一致
3. 构建**布林带**并将带宽解释为波动率度量
4. 从对数收益率计算**已实现波动率**
5. 实现 **MACD**（移动平均收敛/发散指标）
6. 理解每个指标如何映射到生产信号模块

In [ ]:
# ── 环境设置 ──
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import requests

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# ── 加载示例数据 ──
def fetch_btc_hourly(days: int = 60) -> pd.DataFrame:
    """获取 BTC 小时数据用于课程。"""
    end_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_ms = end_ms - days * 86_400_000
    all_rows = []
    cursor = start_ms
    while cursor < end_ms:
        resp = requests.get("https://api.binance.com/api/v3/klines",
                           params={"symbol": "BTCUSDT", "interval": "1h",
                                   "startTime": cursor, "endTime": end_ms, "limit": 1000},
                           timeout=10)
        resp.raise_for_status()
        rows = resp.json()
        if not rows: break
        all_rows.extend(rows)
        cursor = int(rows[-1][0]) + 3_600_000
        if len(rows) < 1000: break
    df = pd.DataFrame(all_rows, columns=["open_time","open","high","low","close","volume",
                                          "close_time","quote_vol","trades","taker_base","taker_quote","ignore"])
    for col in ["open","high","low","close","volume","quote_vol"]:
        df[col] = df[col].astype(float)
    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df = df.set_index("timestamp")[["open","high","low","close","volume","quote_vol"]]
    return df

df = fetch_btc_hourly(60)
close = df["close"]
print(f"✅ 已加载 {len(df)} 根小时K线: {df.index[0].date()} → {df.index[-1].date()}")

---

## 📐 第一节：指数移动平均线 (EMA)

移动平均线通过平滑嘈杂的价格数据来揭示潜在趋势。

### 简单移动平均线 (SMA) — 基准

$$\text{SMA}_k = \frac{1}{k} \sum_{i=0}^{k-1} P_{t-i}$$

所有 $k$ 个价格权重相等。问题：SMA 具有**滞后性** — 需要 $k$ 个周期才能完全反映价格变化。

### EMA — 指数加权

$$\text{EMA}_t = \alpha \cdot P_t + (1 - \alpha) \cdot \text{EMA}_{t-1}$$

其中平滑系数为：

$$\alpha = \frac{2}{k + 1}$$

| $k$ | $\alpha$ | 半衰期 | 特性 |
|-----|----------|--------|------|
| 10  | 0.182    | ~6 根  | 快速，噪声大 |
| 20  | 0.095    | ~13 根 | 中速（机器人的快速 EMA） |
| 50  | 0.039    | ~34 根 | 慢速，平滑（机器人的慢速 EMA） |

机器人使用 **EMA-20** 和 **EMA-50** 进行制度检测（见 Notebook 06）。

In [ ]:
# ── 从零实现 EMA，然后用 pandas ──

# 方法 1：手动循环（理解数学原理）
def ema_manual(prices: pd.Series, span: int) -> pd.Series:
    """使用递推公式逐步计算 EMA。"""
    alpha = 2.0 / (span + 1)
    result = np.empty(len(prices))
    result[0] = prices.iloc[0]  # 用第一个价格初始化
    for i in range(1, len(prices)):
        result[i] = alpha * prices.iloc[i] + (1 - alpha) * result[i - 1]
    return pd.Series(result, index=prices.index, name=f"EMA_{span}")

# 方法 2：Pandas 内置（生产方式）
def ema_pandas(prices: pd.Series, span: int) -> pd.Series:
    """使用 pandas 优化的 ewm() 计算 EMA — 生产代码的方式。"""
    return prices.ewm(span=span, adjust=False).mean()

# 对比两种方法
ema20_manual = ema_manual(close, 20)
ema20_pandas = ema_pandas(close, 20)

# 结果应该完全一致
max_diff = (ema20_manual - ema20_pandas).abs().max()
print(f"手动方法与 pandas 方法的最大差异: {max_diff:.2e}")
print(f"（应该约为 0 或浮点精度误差）")

In [ ]:
# ── 可视化 EMA 交叉 ──
ema20 = ema_pandas(close, 20)
ema50 = ema_pandas(close, 50)

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(close.index, close, label='收盘价', alpha=0.5, linewidth=0.8)
ax.plot(ema20.index, ema20, label='EMA-20（快线）', linewidth=2, color='#2196F3')
ax.plot(ema50.index, ema50, label='EMA-50（慢线）', linewidth=2, color='#FF9800')

# 标记交叉点
cross_up = (ema20 > ema50) & (ema20.shift(1) <= ema50.shift(1))
cross_down = (ema20 < ema50) & (ema20.shift(1) >= ema50.shift(1))
ax.scatter(close.index[cross_up], close[cross_up], marker='^', c='green', s=100, zorder=5, label='看涨交叉')
ax.scatter(close.index[cross_down], close[cross_down], marker='v', c='red', s=100, zorder=5, label='看跌交叉')

ax.set_title('BTC/USDT — EMA 交叉（制度检测的基础）', fontsize=14)
ax.set_ylabel('价格 (USDT)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n📌 机器人的 regime_detector.py 正是使用 EMA-20/EMA-50 交叉")
print(f"   EMA-20 > EMA-50 → 看涨信号")
print(f"   EMA-20 < EMA-50 → 看跌信号")

---

## 📐 第二节：RSI（相对强弱指数）

RSI 在 0–100 的范围内衡量**动量**。它回答的问题是：*
*

### 数学推导

1. 计算价格变化: $\Delta_t = P_t - P_{t-1}$
2. 分离涨跌:
   - $G_t = \max(\Delta_t, 0)$（涨幅）
   - $L_t = \max(-\Delta_t, 0)$（跌幅）
3. 用 EWM 平滑:
   - $\overline{G} = \text{EWM}(G, \text{span}=k)$
   - $\overline{L} = \text{EWM}(L, \text{span}=k)$
4. 相对强度: $RS = \overline{G} / \overline{L}$
5. RSI 公式: $\text{RSI} = 100 - \frac{100}{1 + RS}$

### 解读

| RSI 范围 | 市场状态 | 机器人的操作 |
|---------|---------|----------|
| > 70 | 超买 | 动量信号降低权重 |
| 30–70 | 中性 | 正常运行 |
| < 30 | 超卖 | 均值回归信号激活 |
| < 45 | 机器人阈值 | `momentum.py` 过滤掉弱势资产 |

In [ ]:
# ── 从零实现 RSI ──

def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    """使用 EWM 方法计算 RSI。
    
    这与 bot/signals/momentum.py 中的方法完全相同：
    calculate_rsi(close_prices, period=14)
    """
    # 第1步：价格变化
    delta = prices.diff()
    
    # 第2步：分离涨跌
    gains = delta.clip(lower=0)      # 只保留正变化
    losses = (-delta).clip(lower=0)  # 负变化的绝对值
    
    # 第3步：指数加权移动平均
    avg_gain = gains.ewm(span=period, adjust=False).mean()
    avg_loss = losses.ewm(span=period, adjust=False).mean()
    
    # 第4步：相对强度
    rs = avg_gain / avg_loss
    
    # 第5步：RSI 公式
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

rsi_14 = calculate_rsi(close, 14)

print(f"当前 RSI(14): {rsi_14.iloc[-1]:.1f}")
print(f"最小 RSI: {rsi_14.min():.1f}")
print(f"最大 RSI: {rsi_14.max():.1f}")

In [ ]:
# ── 与生产代码 RSI 对比 ──
from bot.signals.momentum import calculate_rsi as prod_rsi

rsi_prod = prod_rsi(close, 14)
max_diff = (rsi_14 - rsi_prod).abs().max()
print(f"与生产代码 RSI 的最大差异: {max_diff:.2e}")
print("✅ 我们的实现与生产代码完全匹配！")

In [ ]:
# ── RSI 可视化：超买/超卖区域 ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[2, 1], sharex=True)

# 价格
ax1.plot(close.index, close, label='BTC 收盘价', color='#333', linewidth=1)
ax1.set_ylabel('价格 (USDT)')
ax1.set_title('BTC/USDT 价格与 RSI(14)', fontsize=14)
ax1.legend()

# RSI
ax2.plot(rsi_14.index, rsi_14, label='RSI(14)', color='#9C27B0', linewidth=1.2)
ax2.axhline(70, color='red', linestyle='--', alpha=0.7, label='超买 (70)')
ax2.axhline(30, color='green', linestyle='--', alpha=0.7, label='超卖 (30)')
ax2.axhline(45, color='orange', linestyle=':', alpha=0.7, label='机器人阈值 (45)')
ax2.fill_between(rsi_14.index, 70, 100, alpha=0.1, color='red')
ax2.fill_between(rsi_14.index, 0, 30, alpha=0.1, color='green')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

---

## 📐 第三节：布林带 (Bollinger Bands)

布林带在价格周围创建一个**动态包络**，随波动率扩张和收缩。

### 数学公式

$$\text{中轨} = \text{SMA}_k(P)$$

$$\text{上轨} = \text{SMA}_k + n_\sigma \cdot \sigma_k$$

$$\text{下轨} = \text{SMA}_k - n_\sigma \cdot \sigma_k$$

其中 $\sigma_k$ 是最近 $k$ 个价格的滚动标准差。

我们的机器人使用 $k = 20$，$n_\sigma = 2.0$（来自 `config/strategy_params.yaml`）。

### 关键信号

- **价格触及下轨** → 潜在**超卖**（均值回归买入信号）
- **价格触及上轨** → 潜在**超买**
- **带宽收窄**（squeeze）→ 低波动率，可能即将突破
- **带宽扩张** → 高波动率，趋势进行中

In [ ]:
# ── 从零实现布林带 ──

def bollinger_bands(prices: pd.Series, period: int = 20, num_std: float = 2.0):
    """计算布林带。
    
    生产代码参考: bot/signals/mean_reversion.py → build_mean_reversion_frame()
    配置: strategy_params.yaml → mean_reversion.bb_period=20, bb_std=2.0
    """
    middle = prices.rolling(period).mean()
    std = prices.rolling(period).std()
    upper = middle + num_std * std
    lower = middle - num_std * std
    
    # 带宽：标准化波动率度量
    band_width = (upper - lower) / middle
    
    # %B：价格在带内的位置（0 = 下轨, 1 = 上轨）
    pct_b = (prices - lower) / (upper - lower)
    
    return pd.DataFrame({
        'middle': middle,
        'upper': upper,
        'lower': lower,
        'band_width': band_width,
        'pct_b': pct_b,
    }, index=prices.index)

bb = bollinger_bands(close, 20, 2.0)
print(f"当前 %B: {bb['pct_b'].iloc[-1]:.3f}")
print(f"  0.0 = 在下轨, 1.0 = 在上轨")
print(f"  当前带宽: {bb['band_width'].iloc[-1]:.4f}")

In [ ]:
# ── 布林带可视化 ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[3, 1], sharex=True)

# 价格与布林带
ax1.plot(close.index, close, label='收盘价', color='#333', linewidth=1)
ax1.plot(bb['middle'].index, bb['middle'], label='SMA(20)', color='#2196F3', linewidth=1)
ax1.plot(bb['upper'].index, bb['upper'], label='上轨 (+2σ)', color='red', linewidth=0.8, linestyle='--')
ax1.plot(bb['lower'].index, bb['lower'], label='下轨 (-2σ)', color='green', linewidth=0.8, linestyle='--')
ax1.fill_between(bb.index, bb['upper'], bb['lower'], alpha=0.1, color='blue')

# 标记触碰点
touch_lower = close <= bb['lower']
touch_upper = close >= bb['upper']
ax1.scatter(close.index[touch_lower], close[touch_lower], c='green', s=30, zorder=5, label='触碰下轨')
ax1.scatter(close.index[touch_upper], close[touch_upper], c='red', s=30, zorder=5, label='触碰上轨')

ax1.set_title('BTC/USDT — 布林带 (20, 2.0)', fontsize=14)
ax1.set_ylabel('价格 (USDT)')
ax1.legend(fontsize=9)

# 带宽
ax2.fill_between(bb.index, 0, bb['band_width'], alpha=0.3, color='purple')
ax2.plot(bb['band_width'].index, bb['band_width'], color='purple', linewidth=1)
ax2.set_ylabel('带宽')
ax2.set_title('布林带宽度（波动率度量）', fontsize=11)

plt.tight_layout()
plt.show()

---

## 📐 第四节：已实现波动率

**波动率**是价格波动的强度。机器人用它来：

- **制度检测**：高波动率 → 熊市/震荡制度
- **风险预算**：仓位大小与波动率成反比

### 数学公式

1. 对数收益率: $r_t = \ln(P_t / P_{t-1})$
2. 滚动标准差: $\sigma_k = \text{std}(r_{t-k+1}, \ldots, r_t)$
3. 年化: $\sigma_{\text{ann}} = \sigma_k \cdot \sqrt{N}$，其中 $N = 8{,}760$（小时数据：365 × 24）

机器人的制度检测器使用**波动率阈值**来区分平静市场和动荡市场。

In [ ]:
# ── 从零计算已实现波动率 ──

# 第1步：对数收益率
log_returns = np.log(close / close.shift(1))

# 第2步：滚动波动率（20 周期）
vol_20 = log_returns.rolling(20).std()

# 第3步：年化（小时数据 → 每年 8760 小时）
vol_annualized = vol_20 * np.sqrt(8760)

print(f"当前小时波动率: {vol_20.iloc[-1]:.6f}")
print(f"年化波动率:     {vol_annualized.iloc[-1]:.1%}")
print(f"平均年化波动率: {vol_annualized.mean():.1%}")

# 机器人的 regime_detector.py 将此与阈值比较（默认: 0.02 小时波动率）
VOL_THRESHOLD = 0.02
print(f"\n机器人波动率阈值: {VOL_THRESHOLD}")
print(f"当前 vs 阈值: {'高波动率 ⚠️' if vol_20.iloc[-1] > VOL_THRESHOLD else '正常波动率 ✅'}")

In [ ]:
# ── 波动率可视化 ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), height_ratios=[2, 1], sharex=True)

ax1.plot(close.index, close, color='#333', linewidth=1)
ax1.set_ylabel('价格 (USDT)')
ax1.set_title('BTC/USDT — 价格与滚动波动率', fontsize=14)

ax2.fill_between(vol_20.index, 0, vol_20, alpha=0.3, color='orange')
ax2.plot(vol_20.index, vol_20, color='darkorange', linewidth=1)
ax2.axhline(VOL_THRESHOLD, color='red', linestyle='--', label=f'制度阈值 ({VOL_THRESHOLD})')
ax2.set_ylabel('小时波动率 (σ₂₀)')
ax2.legend()

plt.tight_layout()
plt.show()

---

## 📐 第五节：MACD（移动平均收敛/发散指标）

MACD 结合两条 EMA 来检测**趋势变化和动量转换**。

### 数学公式

$$\text{MACD 线} = \text{EMA}_{12}(P) - \text{EMA}_{26}(P)$$

$$\text{信号线} = \text{EMA}_9(\text{MACD 线})$$

$$\text{柱状图} = \text{MACD 线} - \text{信号线}$$

### 信号解读

- **MACD 上穿信号线** → 看涨动量
- **MACD 下穿信号线** → 看跌动量
- **柱状图增长** → 趋势增强
- **柱状图缩减** → 趋势减弱

In [ ]:
# ── 从零实现 MACD ──

def compute_macd(prices: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    """计算 MACD 线、信号线和柱状图。"""
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    
    return pd.DataFrame({
        'macd': macd_line,
        'signal': signal_line,
        'histogram': histogram,
    }, index=prices.index)

macd = compute_macd(close)
print(f"当前 MACD: {macd['macd'].iloc[-1]:.2f}")
print(f"信号线:    {macd['signal'].iloc[-1]:.2f}")
print(f"柱状图:    {macd['histogram'].iloc[-1]:.2f}")

In [ ]:
# ── MACD 可视化 ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[2, 1], sharex=True)

ax1.plot(close.index, close, color='#333', linewidth=1)
ax1.set_ylabel('价格 (USDT)')
ax1.set_title('BTC/USDT — MACD', fontsize=14)

ax2.plot(macd['macd'].index, macd['macd'], label='MACD', color='#2196F3', linewidth=1.5)
ax2.plot(macd['signal'].index, macd['signal'], label='信号线', color='#FF9800', linewidth=1.5)
colors = ['#26a69a' if v >= 0 else '#ef5350' for v in macd['histogram']]
ax2.bar(macd.index, macd['histogram'], color=colors, alpha=0.5, width=0.03)
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.legend()
ax2.set_ylabel('MACD')

plt.tight_layout()
plt.show()

---

## 💻 第六节：生产代码指标框架

机器人的 `mean_reversion.py` 在一次调用中构建完整的指标 DataFrame。让我们使用它，看看所有指标如何协同工作：

In [ ]:
# ── 生产代码均值回归指标框架 ──
from bot.signals.mean_reversion import build_mean_reversion_frame

# 生产代码函数在一次向量化运算中计算 RSI + 布林 %B
mr_frame = build_mean_reversion_frame(close, rsi_period=14, bb_period=20, bb_std=2.0)

print("生产代码指标框架的列:")
print(mr_frame.columns.tolist())
print(f"\n最新值:")
mr_frame.tail(3)

---

## 📐 总结：指标 → 机器人模块映射

| 指标 | 公式 | 机器人模块 | 用途 |
|------|------|-----------|------|
| EMA-20/50 | $\alpha P_t + (1-\alpha) \text{EMA}_{t-1}$ | `regime_detector.py` | 牛/熊市分类 |
| RSI(14) | $100 - 100/(1+RS)$ | `momentum.py` | 资产强度过滤 |
| 布林 %B | $(P - L) / (U - L)$ | `mean_reversion.py` | 超卖检测 |
| 波动率 | $\text{std}(\ln P_t/P_{t-1})$ | `regime_detector.py` | 高波动制度标记 |
| MACD | $\text{EMA}_{12} - \text{EMA}_{26}$ | 辅助参考 | 趋势确认 |

---

## 🔬 练习

### 练习 1：多资产 RSI 仪表板 🔬

获取 3 种资产（BTC、ETH、SOL）的小时数据，分别计算 RSI(14)，在同一图表上绘制。哪种资产当前最超买/超卖？

In [ ]:
# ── 练习 1：在这里写代码 ──

# 你的代码

### 练习 2：带宽百分位 ⭐

计算当前布林带宽相对于过去 60 天的百分位数。非常低的百分位（< 10%）表示
— 可能即将突破。

In [ ]:
# ── 练习 2：在这里写代码 ──

# 提示：使用 scipy.stats.percentileofscore()
# 你的代码

### 练习 3：自定义复合指标 ⭐⭐

创建一个将 RSI 和布林 %B 组合成单一评分的复合指标，范围从 -1（极度超卖）到 +1（极度超买）。在 BTC 数据上测试。

In [ ]:
# ── 练习 3：在这里写代码 ──

# 你的代码

---

## ✅ 知识检查

1. 为什么 EMA 比 SMA 对价格变化反应更快？
2. RSI 值为多少表示近期涨跌
？
3. 如果布林带正在收窄，这说明波动率怎样？
4. 为什么计算波动率时使用**对数收益率**而非简单收益率？
5. 标准 MACD 设置中的三条 EMA 分别是多少？

<details>
<summary>点击查看答案</summary>

1. EMA 给近期价格更高的权重（指数衰减权重），因此新数据的影响更大
2. RSI = 50 表示平均涨幅等于平均跌幅
3. 波动率在下降 — 这是一个
，通常预示着即将到来的突破行情
4. 对数收益率在时间上可加，且近似正态分布，使统计分析更严谨
5. EMA-12（快线）、EMA-26（慢线）和 EMA-9（信号线）

</details>

---

## 🔗 下一步：Notebook 03 — 风险指标深度解析

你现在知道如何衡量市场在做什么。下一步是衡量**你的策略表现如何** — 使用夏普比率、索提诺比率和卡尔玛比率。在 Notebook 03 中，我们将从第一性原理推导这些指标，包括用 Delta 方法估计标准误差。

**打开：** `03_风险指标深度解析.ipynb`